# Build combined k-means input matrices (Quartet multiomics)

Reads the per-modality corrected outputs written by `evaluation_data/multiomics/03_central_RBE.ipynb`
and the uncorrected inputs from `before/`, then stacks the three omics layers into a single
joint matrix per condition (before / central-corrected / FedRBE-corrected).

There is no true shared library ID across the Quartet omics layers. Samples are pseudo-matched
by `pseudo_sample = "{client}_{donor}_{rep}_{i}"` (built in `02_prepare_RBE_inputs.ipynb`).
Each modality block is row z-scored and divided by `sqrt(number_of_features)` so no modality
dominates the joint Euclidean distance by feature count.

**Reads:**
- `evaluation_data/multiomics/before/<Modality>/central_intensities_log_UNION.tsv` + `metadata.tsv`
- `evaluation_data/multiomics/after/<Modality>/intensities_log_Rcorrected_UNION.tsv` + `metadata.tsv`
- `evaluation_data/multiomics/after/<Modality>/FedApp_corrected_data.tsv` (written by `04_run_fedrbe.ipynb`, optional)

**Writes:**
- `evaluation_data/multiomics/after/all_modalities_before_kmeans_matrix.tsv`
- `evaluation_data/multiomics/after/all_modalities_corrected_kmeans_matrix.tsv`
- `evaluation_data/multiomics/after/all_modalities_fedsim_kmeans_matrix.tsv` (if FedRBE outputs exist)
- `evaluation_data/multiomics/after/all_modalities_metadata.tsv`

**Consumed by:** the standard real-dataset flow in this repo (notebooks `../01_data_preparation.ipynb` … `../04_analysis_metrics_plots.ipynb`).


In [ ]:
library(tidyverse)

MULTIOMICS_DIR <- if (dir.exists("figshare_data")) "." else "evaluation_data/multiomics"
BEFORE_DIR <- file.path(MULTIOMICS_DIR, "before")
AFTER_DIR  <- file.path(MULTIOMICS_DIR, "after")

MODALITIES <- c("Transcriptomics", "Proteomics", "Metabolomics")
DONOR_LEVELS <- c("D6", "D5", "F7", "M8")

## Helper functions

In [ ]:
read_feature_matrix <- function(path) {
  df <- readr::read_tsv(path, show_col_types = FALSE)
  first_col <- names(df)[1]
  mat <- df %>%
    column_to_rownames(first_col) %>%
    as.data.frame(check.names = FALSE)
  mat[] <- lapply(mat, as.numeric)
  as.matrix(mat)
}

write_feature_matrix <- function(expr, path) {
  out <- as.data.frame(expr, check.names = FALSE) %>% rownames_to_column("rowname")
  write.table(out, file = path, sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)
}

row_zscore <- function(expr) {
  values <- as.matrix(expr)
  means <- rowMeans(values, na.rm = TRUE)
  sds <- apply(values, 1, sd, na.rm = TRUE)
  sds[!is.finite(sds) | sds == 0] <- 1
  sweep(sweep(values, 1, means, "-"), 1, sds, "/")
}

# Reindex columns to pseudo_sample, row-zscore, then divide by sqrt(n_features)
# so each modality contributes equally to the joint Euclidean distance
# regardless of feature count.
build_modality_block <- function(expr, metadata, modality, sample_keys) {
  mat <- expr[, metadata$file, drop = FALSE]
  colnames(mat) <- metadata$pseudo_sample
  if (anyDuplicated(colnames(mat)) > 0) stop(modality, ": duplicate pseudo-sample columns.", call. = FALSE)
  mat <- mat[, sample_keys, drop = FALSE]
  scaled <- row_zscore(mat) / sqrt(nrow(mat))
  rownames(scaled) <- paste0(modality, "__", make.unique(rownames(scaled)))
  scaled
}

## Load per-modality matrices

In [ ]:
before_matrices    <- list()
corrected_matrices <- list()
fedrbe_matrices    <- list()
metadata_by_modality <- list()

for (modality in MODALITIES) {
  before_dir  <- file.path(BEFORE_DIR, modality)
  after_dir   <- file.path(AFTER_DIR,  modality)

  before_matrices[[modality]] <- read_feature_matrix(
    file.path(before_dir, "central_intensities_log_UNION.tsv")
  )
  corrected_matrices[[modality]] <- read_feature_matrix(
    file.path(after_dir, "intensities_log_Rcorrected_UNION.tsv")
  )

  fedrbe_path <- file.path(after_dir, "FedApp_corrected_data.tsv")
  if (file.exists(fedrbe_path)) {
    fedrbe_matrices[[modality]] <- read_feature_matrix(fedrbe_path)
  }

  metadata_by_modality[[modality]] <- readr::read_tsv(
    file.path(after_dir, "metadata.tsv"), show_col_types = FALSE
  ) %>% mutate(
    condition = factor(condition, levels = DONOR_LEVELS),
    batch = factor(batch),
    rep = as.integer(rep)
  )
}

message("Loaded modalities: ", paste(MODALITIES, collapse = ", "))
message("FedRBE outputs available: ", length(fedrbe_matrices) == length(MODALITIES))


## Build combined matrices

Verify that all three modalities share the same `pseudo_sample` keys, then stack
the row-zscored + sqrt-normalised modality blocks into joint matrices.

In [ ]:
sample_keys <- sort(unique(metadata_by_modality[[MODALITIES[1]]]$pseudo_sample))

for (modality in MODALITIES) {
  keys <- sort(unique(metadata_by_modality[[modality]]$pseudo_sample))
  if (!identical(keys, sample_keys)) stop(modality, ": pseudo-sample keys do not match across modalities.", call. = FALSE)
}

before_blocks    <- purrr::map(MODALITIES, ~ build_modality_block(before_matrices[[.x]],    metadata_by_modality[[.x]], .x, sample_keys))
corrected_blocks <- purrr::map(MODALITIES, ~ build_modality_block(corrected_matrices[[.x]], metadata_by_modality[[.x]], .x, sample_keys))

names(before_blocks)    <- MODALITIES
names(corrected_blocks) <- MODALITIES

before_kmeans_matrix    <- do.call(rbind, before_blocks)
corrected_kmeans_matrix <- do.call(rbind, corrected_blocks)

if (length(fedrbe_matrices) == length(MODALITIES)) {
  fedrbe_blocks  <- purrr::map(MODALITIES, ~ build_modality_block(fedrbe_matrices[[.x]], metadata_by_modality[[.x]], .x, sample_keys))
  names(fedrbe_blocks) <- MODALITIES
  fedrbe_kmeans_matrix <- do.call(rbind, fedrbe_blocks)
} else {
  fedrbe_kmeans_matrix <- NULL
  message("Skipping FedRBE matrix — not all modalities have fedrbe outputs.")
}

stopifnot(!anyNA(before_kmeans_matrix), all(is.finite(before_kmeans_matrix)))
stopifnot(!anyNA(corrected_kmeans_matrix), all(is.finite(corrected_kmeans_matrix)))
stopifnot(anyDuplicated(rownames(before_kmeans_matrix)) == 0)
stopifnot(anyDuplicated(rownames(corrected_kmeans_matrix)) == 0)

## Build combined metadata

In [ ]:
combined_metadata <- metadata_by_modality[[MODALITIES[1]]] %>%
  select(pseudo_sample, client, condition, rep) %>%
  distinct()

for (modality in MODALITIES) {
  modality_meta <- metadata_by_modality[[modality]] %>%
    select(pseudo_sample, file, batch, lab, platform, protocol, date) %>%
    rename_with(~ paste0(modality, "_", .x), -pseudo_sample)
  combined_metadata <- combined_metadata %>% left_join(modality_meta, by = "pseudo_sample")
}

combined_metadata <- combined_metadata %>%
  mutate(pseudo_sample = factor(pseudo_sample, levels = sample_keys)) %>%
  arrange(pseudo_sample) %>%
  mutate(pseudo_sample = as.character(pseudo_sample))

stopifnot(identical(colnames(before_kmeans_matrix),    combined_metadata$pseudo_sample))
stopifnot(identical(colnames(corrected_kmeans_matrix), combined_metadata$pseudo_sample))
if (!is.null(fedrbe_kmeans_matrix)) {
  stopifnot(identical(colnames(fedrbe_kmeans_matrix), combined_metadata$pseudo_sample))
}

## Write outputs

In [ ]:
write_feature_matrix(before_kmeans_matrix,    file.path(AFTER_DIR, "all_modalities_before_kmeans_matrix.tsv"))
write_feature_matrix(corrected_kmeans_matrix, file.path(AFTER_DIR, "all_modalities_corrected_kmeans_matrix.tsv"))
if (!is.null(fedrbe_kmeans_matrix)) {
  write_feature_matrix(fedrbe_kmeans_matrix, file.path(AFTER_DIR, "all_modalities_fedsim_kmeans_matrix.tsv"))
}
write.table(combined_metadata, file = file.path(AFTER_DIR, "all_modalities_metadata.tsv"),
            sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

tibble(
  matrix = c("before", "corrected", if (!is.null(fedrbe_kmeans_matrix)) "fedrbe" else character(0)),
  features = c(nrow(before_kmeans_matrix), nrow(corrected_kmeans_matrix),
               if (!is.null(fedrbe_kmeans_matrix)) nrow(fedrbe_kmeans_matrix) else integer(0)),
  samples = c(ncol(before_kmeans_matrix), ncol(corrected_kmeans_matrix),
              if (!is.null(fedrbe_kmeans_matrix)) ncol(fedrbe_kmeans_matrix) else integer(0))
)

## Write per-client `design.tsv` + `intensities.tsv` slices

Drop a flat per-client layout under `evaluation_data/multiomics/before/<client>/`
so the standard `evaluation_clusterization_after_correction/real_datasets/01-04`
notebooks (which use `dataset_configs()` + `discover_clients()`) treat
multiomics like any other dataset.

Each `<client>/` directory holds a per-client column slice of the joint
**before** k-means matrix (already row-zscored within modality and equal
block-weighted) plus a `design.tsv` with `file`, `condition`, and `rep`.

In [ ]:
clients <- sort(unique(combined_metadata$client))
message("Per-client splits for ", length(clients), " client(s).")

for (cl in clients) {
  cl_meta <- combined_metadata %>%
    filter(client == cl) %>%
    arrange(pseudo_sample) %>%
    transmute(file = pseudo_sample, condition = condition, rep = rep)

  cl_dir <- file.path(BEFORE_DIR, cl)
  dir.create(cl_dir, recursive = TRUE, showWarnings = FALSE)

  write.table(cl_meta, file = file.path(cl_dir, "design.tsv"),
              sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

  cl_mat <- before_kmeans_matrix[, cl_meta$file, drop = FALSE]
  out_df <- as.data.frame(cl_mat, check.names = FALSE) %>% rownames_to_column("rowname")
  write.table(out_df, file = file.path(cl_dir, "intensities.tsv"),
              sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)
}

## Session information

In [ ]:
sessionInfo()